# 03 · Case–control comparison of the BIODEP gene panel

**Purpose.** Test whether the 16 BIODEP candidate genes differ between MDD cases and healthy controls in GSE98793.

## Analysis plan (written before looking at group differences)

**Primary analysis.** One linear regression per gene: expression (log2) ~ group + batch + sex + age. The group coefficient is the adjusted difference between all 128 cases and 64 controls. P-values are corrected across the 16 genes with the Benjamini–Hochberg false discovery rate (FDR); FDR < 0.05 is considered significant.

**Expected directions (Cattaneo et al. 2020; Sforzini et al. 2023).** Higher in cases: IL1B, IL6, TNF, MIF, P2RX7, CCL2, CXCL12, A2M, STAT1, FKBP5, SGK1. Lower in cases: NR3C1. No difference (negative controls): AQP4, ISG15, USP18. CRP is reported without a strong prior.

**Secondary analyses.** (1) The three scores (two inflammation scores, glucocorticoid resistance), same model. (2) Three groups: controls, cases without anxiety, cases with anxiety.

**Known limits.** Six genes are flagged as low expression on this array (AQP4, CCL2, CRP, CXCL12, IL6, USP18). Medication status and CRP are not available. All 192 samples are used; Leday et al. (2018) analysed 170 after quality control without listing exclusions.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

samples = pd.read_csv("../data/processed/GSE98793_samples.csv", index_col=0)
genes = pd.read_csv("../data/processed/GSE98793_panel_expression.csv", index_col=0)
scores = pd.read_csv("../data/processed/GSE98793_scores.csv", index_col=0)

data = samples.join(genes).join(scores)
data["mdd"] = data["group"].str.startswith("CASE").astype(int)

print(data.shape)
print(data.groupby(["mdd", "anxiety"]).size())

(192, 26)
mdd  anxiety
0    no         64
1    no         64
     yes        64
dtype: int64


## 1. Primary analysis: one regression per gene

In [2]:
gene_names = genes.columns.tolist()
low_expression = ["AQP4", "CCL2", "CRP", "CXCL12", "IL6", "USP18"]  # flagged in notebook 02

rows = []
for gene in gene_names:
    model = smf.ols(f"{gene} ~ mdd + C(batch) + C(sex) + age", data=data).fit()
    rows.append({
        "gene": gene,
        "difference": model.params["mdd"],
        "std_difference": model.params["mdd"] / model.resid.std(),
        "p_value": model.pvalues["mdd"],
    })

results = pd.DataFrame(rows).set_index("gene")
results["fdr"] = multipletests(results["p_value"], method="fdr_bh")[1]
results["low_expression"] = results.index.isin(low_expression)

results.sort_values("p_value").round(3)

,difference,std_difference,p_value,fdr,low_expression
gene,,,,,
FKBP5,0.208,0.559,0.000,0.006,False
TNF,0.126,0.380,0.015,0.119,False
NR3C1,-0.041,-0.212,0.173,0.595,False
AQP4,-0.057,-0.210,0.176,0.595,True
ISG15,0.123,0.169,0.275,0.595,False
CXCL12,0.044,0.144,0.355,0.595,True
CCL2,0.049,0.138,0.374,0.595,True
STAT1,0.049,0.137,0.378,0.595,False
MIF,-0.053,-0.128,0.409,0.595,False


**Result.** One of the 16 genes differs between cases and controls after FDR correction: FKBP5 is higher in MDD (difference 0.21 log2 units, about 15% higher; standardised difference 0.56; FDR = 0.006), in the direction reported by BIODEP. TNF is higher in cases with a nominal p-value of 0.015 but does not survive correction (FDR = 0.12). NR3C1 is lower in cases, as predicted, but the effect is small and not significant (standardised difference −0.21, p = 0.17). The other well-expressed genes (IL1B, MIF, P2RX7, STAT1, A2M, SGK1) show standardised differences near 0.1. The three negative controls (AQP4, ISG15, USP18) show no difference, as expected.

**Interpretation.** The replication is partial: the glucocorticoid-related finding for FKBP5 replicates, while the inflammation-related findings do not. Non-replication here does not show absence of an effect: this sample has limited power for small effects, medication status is unknown, and four inflammation genes are poorly measured on this array.